# Module 2 — Working with the Data (in Pandas)

Homework: [cohorts/2026/homework2.md](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework2.md)
Submit: [courses.datatalks.club/sma-zoomcamp-2026/homework/hw02](https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw02)

Kernel: **Python (stock-markets-zoomcamp)** — the shared repo venv.

| Q | Question | Answer |
| --- | --- | --- |
| 1 | Total withdrawn IPO value of the largest company class | **$500M** (Acquisition Corp, $499.99M) |
| 2 | Median Sharpe ratio on 2026-09-11 for pre-Sep-2025 IPOs | **0.04** (computed 0.0501) |
| 3 | Holding period maximising median growth | **1 month** (median 0.935) |
| 4 | Net income from the RSI < 30 strategy | **$65k** ($65,805 over 5,206 trades) |
| 5 | How to make an IPO strategy profitable | **`RSI<30 & natr>3 & slowk<20`** — +62.7% profit at equal capital, 78.1% alpha (free text) |

> ⚠️ **The two iposcoop tables are live and change daily.** This notebook was run on
> **2026-09-18**, a week after the homework's 2026-09-11 anchor date, so raw row counts
> drift from the ones quoted in the task. Where that happens it is called out in the
> cell below the number. None of the drift changes a multiple-choice answer.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import io

import numpy as np
import pandas as pd
import requests

import smaz
from smaz import data, features, utils

utils.set_plot_defaults()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

AS_OF = pd.Timestamp("2026-09-11")   # the date the homework anchors on

# iposcoop blocks the default pandas/requests user-agent, exactly like Wikipedia in module 1.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}


def read_html_table(url: str, index: int = 0) -> pd.DataFrame:
    """`pd.read_html` behind a browser user-agent."""
    response = requests.get(url, headers=HEADERS, timeout=60)
    response.raise_for_status()
    return pd.read_html(io.StringIO(response.text))[index]


print("smaz", smaz.__version__, "| repo root:", smaz.config.REPO_ROOT)

---

## Question 1 — [IPO] Withdrawn IPOs by company type

> **What is the total withdrawn IPO value (in $ millions) for the company class with the
> highest total withdrawal value?**
>
> From the [recently-filed IPO list](https://www.iposcoop.com/ipos-recently-filed/), find
> which company type saw the most withdrawn IPO value before Sep 11, 2026.

### Load the recently-filed table

In [ ]:
ipos_recent = read_html_table("https://www.iposcoop.com/ipos-recently-filed/")

print(ipos_recent.shape)
print(ipos_recent["Expected To Trade"].value_counts().to_string())
ipos_recent.head()

### Keep the withdrawn deals filed before the anchor date

The task expects **32** withdrawn rows. The live table now shows 34 — two more IPOs
(C2 Capital Group, OTSAW) were withdrawn in the week *after* the homework's 2026-09-11
cut-off. Filtering `File Date < 2026-09-11` reproduces the expected 32 exactly, so the
extra rows are genuinely new data rather than a parsing difference.

In [ ]:
withdrawn = ipos_recent[ipos_recent["Expected To Trade"] == "Withdrawn"].copy()
withdrawn["File Date"] = pd.to_datetime(withdrawn["File Date"])

print("withdrawn in the live table:", len(withdrawn))

withdrawn = withdrawn[withdrawn["File Date"] < AS_OF].copy()
print("withdrawn filed before", AS_OF.date(), ":", len(withdrawn))

### Classify the company type

The rules are applied **in order** and the first match wins, which is what makes
`EUPEC International Group Ltd.` a `Group` rather than a `Limited`. Matching is a plain
case-sensitive substring test, so `Xinxu Copper Industry **Technology** Ltd.` does *not*
hit the `Technologies` rule and falls through to `Limited`.

In [ ]:
COMPANY_TYPE_RULES = [
    ("Technologies", ["Technologies"]),
    ("Acquisition Corp", ["Acquisition Corp", "Acquisition Corporation", "Corp"]),
    ("Inc.", ["Inc", "Incorporated"]),
    ("Group", ["Group"]),
    ("Limited", ["Ltd", "Limited"]),
    ("Holdings", ["Holdings", "Holding"]),
]


def classify_company(name: str) -> str:
    """First matching rule wins -- the ordering above is part of the specification."""
    for label, patterns in COMPANY_TYPE_RULES:
        if any(p in name for p in patterns):
            return label
    return "Other"


withdrawn["Company Type"] = withdrawn["Company"].apply(classify_company)
print(withdrawn["Company Type"].value_counts().to_string())

# the two worked examples from the task description
for name in ["EUPEC International Group Ltd.", "Xinxu Copper Industry Technology Ltd."]:
    print(f"{name!r:45s} -> {classify_company(name)}")

### Parse prices and volumes

`'$8.00'` -> `8.0`, `'-'` and blanks -> `NaN`. `Avg_price` is the midpoint of the low/high
range (`.mean(axis=1)` skips NaNs, so a row with only one side still gets a price).

In [ ]:
def to_number(value) -> float:
    """'$1,234.50' -> 1234.5 ; '-' / '' / NaN -> NaN."""
    if pd.isna(value):
        return np.nan
    text = str(value).replace("$", "").replace(",", "").strip()
    if text in {"-", ""}:
        return np.nan
    try:
        return float(text)
    except ValueError:
        return np.nan


withdrawn["Avg_price"] = withdrawn[["Price Low", "Price High"]].map(to_number).mean(axis=1)
withdrawn["Shares (millions)"] = withdrawn["Shares (millions)"].map(to_number)
withdrawn["Est $ Vol (millions)"] = withdrawn["Est $ Vol (millions)"].map(to_number)

withdrawn[["Company", "Shares (millions)", "Price Low", "Price High", "Avg_price"]].head()

### Deal value, with the estimated-volume fallback

Six rows carry `Shares (millions) == 0` *and* no price range at all, so the product is
`NaN` and the `Est $ Vol (millions)` column takes over. Note the fallback keys off the
*product* being null, not off the shares being zero — a genuine `0 x price = 0` would be
kept as zero.

In [ ]:
shares_x_price = withdrawn["Shares (millions)"] * withdrawn["Avg_price"]
withdrawn["Shares_offered_value"] = shares_x_price.where(
    shares_x_price.notna(), withdrawn["Est $ Vol (millions)"]
)

print("rows falling back to Est $ Vol:", int(shares_x_price.isna().sum()))
withdrawn[["Company", "Company Type", "Shares_offered_value"]].sort_values(
    "Shares_offered_value", ascending=False
).head(10)

In [ ]:
by_type = (
    withdrawn.groupby("Company Type")["Shares_offered_value"]
    .agg(total="sum", deals="count")
    .sort_values("total", ascending=False)
)
print(by_type.round(2).to_string())

winner = by_type.index[0]
print(f"\nhighest: {winner} -- ${by_type.loc[winner, 'total']:.2f}M")

In [ ]:
ax = by_type["total"].plot(kind="bar", color="#4C78A8", edgecolor="none")
ax.bar(winner, by_type.loc[winner, "total"], color="#E45756")  # highlight the winner
ax.set_title("Withdrawn IPO value by company type (filed before 2026-09-11)")
ax.set_xlabel("company type")
ax.set_ylabel("total value, $M")
for i, v in enumerate(by_type["total"]):
    ax.text(i, v + 6, f"{v:,.0f}", ha="center")

**Answer 1: Acquisition Corp, $499.99M — i.e. the `500` option.**

Five withdrawn SPAC-style shells add up to ~$500M, ahead of `Inc.` ($351M, a single deal —
Clear Street Group) and `Holdings` ($312M). Two observations worth keeping:

- The `Acquisition Corp` bucket wins on **count x uniform ticket size**, not on one big
  deal: SPACs all price at exactly $10.00, and four of the five offered 6–20M units.
  The `Inc.` bucket is a single company that happens to be large.
- The classification is doing real work here. `Helio Corp.` is a $15M uplisting unit deal
  with nothing SPAC-like about it, but the bare `Corp` pattern sweeps it into
  `Acquisition Corp` anyway. Drop it and the bucket is $485M — still the winner, so the
  answer is not sensitive to that edge case.

---

## Question 2 — [IPO] Median Sharpe ratio for 2025 IPOs

> **What is the median Sharpe ratio (as of 11 September 2026) for companies that went
> public before 1 September 2025?**

### Load the 2025 pricings list

In [ ]:
ipos_2025 = read_html_table("https://www.iposcoop.com/2025-pricings/")
ipos_2025["Offer Date"] = pd.to_datetime(ipos_2025["Offer Date"], format="%m/%d/%Y")
ipos_2025["Return_pct"] = ipos_2025["Return"].str.rstrip("%").astype(float)

print("IPOs priced in 2025:", len(ipos_2025))
ipos_2025.head()

### Filter to the pre-September cohort

A `0.00%` return on iposcoop means the *Current Price* column was never updated away from
the first-day close — the ticker is dead, halted or simply untracked, so those rows are
dropped as the task asks.

The task quotes **148** survivors; this run gets **146**. Two more names went stale in the
week since (iposcoop freezes the price rather than deleting the row), which is the same
live-data drift as in Q1. Two tickers out of 146 cannot move a median meaningfully.

In [ ]:
pre_sep = ipos_2025[ipos_2025["Offer Date"] < "2025-09-01"]
print("priced before 2025-09-01:", len(pre_sep))

ipo_universe = pre_sep[pre_sep["Return_pct"] != 0].copy()
print("after dropping 0% returns:", len(ipo_universe), "(task quotes 148)")

tickers = sorted(ipo_universe["Symbol"].unique())
print("unique tickers:", len(tickers))

### Download the daily OHLCV

Tickers are fetched one at a time rather than as a single batch: a third of this universe
is delisted micro-caps, and yfinance's batch mode silently returns an all-NaN block for a
dead ticker instead of reporting which one failed. The whole result is memoised to
`data/cache/` through `smaz.data.cached`, so re-running the notebook does not re-hit Yahoo.

In [ ]:
def download_ipo_ohlcv() -> pd.DataFrame:
    import warnings

    import yfinance as yf

    frames, failed = [], []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for ticker in tickers:
            try:
                hist = yf.download(
                    ticker,
                    start="2025-01-01",
                    end="2026-09-13",
                    auto_adjust=True,
                    progress=False,
                    threads=False,
                )
            except Exception:                      # noqa: BLE001 - yfinance raises broadly
                failed.append(ticker)
                continue
            if hist is None or hist.empty:
                failed.append(ticker)
                continue
            if isinstance(hist.columns, pd.MultiIndex):
                hist.columns = hist.columns.droplevel(1)   # single ticker -> drop the level
            hist = hist.reset_index()
            hist["Ticker"] = ticker
            frames.append(hist)

    print(f"downloaded {len(frames)} tickers, {len(failed)} unavailable: {failed}")
    return pd.concat(frames, ignore_index=True)


stocks_df = data.cached(
    "ipo2025_ohlcv", download_ipo_ohlcv, tickers=tickers, end="2026-09-13"
)
stocks_df = stocks_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(stocks_df.shape)
print("tickers with data:", stocks_df["Ticker"].nunique(), "(task quotes ~134)")
print("date range:", stocks_df["Date"].min().date(), "->", stocks_df["Date"].max().date())
stocks_df.head()

### Feature engineering

Both rolling calculations are `groupby("Ticker").transform(...)`, never a bare
`stocks_df["Close"].rolling(...)`. The frame is a stack of 132 separate price series; a
global rolling window would blend the tail of one ticker into the head of the next.

One caveat on the volatility formula the task prescribes: `Close.rolling(30).std()` is the
standard deviation of the **price**, in dollars, not of returns. So `volatility` scales
with the share price and the resulting `Sharpe` is not a textbook Sharpe ratio (a $200
stock looks far "riskier" than a $2 stock at identical percentage swings). It is
reproduced exactly as specified, but the level of the number is not comparable to a Sharpe
ratio computed from returns.

In [ ]:
close_by_ticker = stocks_df.groupby("Ticker")["Close"]

stocks_df["growth_252d"] = close_by_ticker.transform(lambda s: s / s.shift(252))
stocks_df["volatility"] = close_by_ticker.transform(
    lambda s: s.rolling(30).std() * np.sqrt(252)
)
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

stocks_df[["Date", "Ticker", "Close", "growth_252d", "volatility", "Sharpe"]].tail()

### The 2026-09-11 snapshot

In [ ]:
snapshot = stocks_df[stocks_df["Date"] == AS_OF]
print("stocks trading on", AS_OF.date(), ":", len(snapshot))
print("reached the 252-day milestone:", int(snapshot["growth_252d"].notna().sum()))

snapshot[["growth_252d", "volatility", "Sharpe"]].describe()

In [ ]:
median_sharpe = snapshot["Sharpe"].median()
print(f"median Sharpe      : {median_sharpe:.4f}")
print(f"median growth_252d : {snapshot['growth_252d'].median():.4f}")
print(f"mean   growth_252d : {snapshot['growth_252d'].mean():.4f}")

# a handful of tickers have not moved for 30 sessions -> ~zero price-stdev -> exploding Sharpe
degenerate = snapshot[snapshot["volatility"] < 1e-4].dropna(subset=["Sharpe"])
print("\ntickers with an essentially flat 30-session price:", len(degenerate))
print(
    degenerate[["Ticker", "Close", "growth_252d", "volatility", "Sharpe"]].to_string(index=False)
)
print("\nmean Sharpe including these:", snapshot["Sharpe"].mean())
print("mean Sharpe excluding these:", snapshot.loc[snapshot["volatility"] >= 1e-4, "Sharpe"].mean())

**Answer 2: median Sharpe = 0.0501 → the `0.04` option** (the nearest of the four offered;
`0.1` is twice as far away).

What the describe table actually says:

- **`growth_252d` median 0.59 vs mean 1.06.** The typical 2025 IPO is worth **41% less**
  than a year ago, while the mean sits near break-even — one 33x survivor and a couple of
  10x names drag the average up over 130 stocks. Mean growth is useless here; this is the
  textbook case for the median.
- **130 of 132 stocks cleared the 252-day milestone**, so the sample is not being thinned
  by young listings — the losses are real, not a windowing artifact.
- **Three tickers have an infinite or absurd Sharpe** (EFTY, MAMK, MAGH) purely because
  the underlying price has not moved for 30 consecutive sessions, making the denominator
  zero. This is a data-quality tell, not attractive risk-adjusted returns. The median is
  immune to the distortion; a mean Sharpe is literally `inf`.
- Risk-adjusted ranking does beat raw growth in one respect: the best *finite* Sharpe
  names (HCMAU, CEPF — SPAC units grinding from $10.00 to $10.47) are boring, not the
  high-growth names. A 3.4% move with near-zero variance scores better than a 5x with
  violent swings.

---

## Question 3 — [IPO] Fixed-months holding strategy

> **What is the optimal number of months (1 to 12) to hold a newly IPO'd stock to maximise
> the median growth value?**

12 forward-looking columns, 21 trading days to the month, measured from each stock's own
first close.

In [ ]:
GROWTH_COLS = []
for month in range(1, 13):
    col = f"future_growth_{month}_m"
    stocks_df[col] = stocks_df.groupby("Ticker")["Close"].transform(
        lambda s, days=21 * month: s.shift(-days) / s
    )
    GROWTH_COLS.append(col)

stocks_df[["Date", "Ticker", "Close", *GROWTH_COLS[:3]]].head()

### Isolate each stock's first trading day

The inner join keeps exactly one row per ticker — the IPO day — so every growth column is
now "what a buyer at the first close would have earned".

In [ ]:
min_dates = stocks_df.groupby("Ticker")["Date"].min().reset_index()
ipo_day = min_dates.merge(stocks_df, on=["Ticker", "Date"], how="inner")

print("rows after the inner join:", len(ipo_day))
ipo_day[["Ticker", "Date", "Close", *GROWTH_COLS[:3]]].head()

In [ ]:
horizon_stats = ipo_day[GROWTH_COLS].describe().T[["count", "mean", "50%"]]
horizon_stats.columns = ["count", "mean", "median"]
print(horizon_stats.round(3).to_string())

best_month = ipo_day[GROWTH_COLS].median().idxmax()
best_median = ipo_day[GROWTH_COLS].median().max()
print(f"\nbest: {best_month} -- median growth {best_median:.4f}")

In [ ]:
medians = ipo_day[GROWTH_COLS].median()
medians.index = range(1, 13)

ax = medians.plot(marker="o", color="#4C78A8")
ax.axhline(1.0, color="#E45756", linestyle="--", linewidth=1)
ax.text(11.4, 1.005, "break-even", color="#E45756", ha="right")
ax.set_title("Median growth of a 2025 IPO vs holding period (bought at the first close)")
ax.set_xlabel("holding period, months")
ax.set_ylabel("median growth multiple")
ax.set_xticks(range(1, 13))

### Robustness: the mean column is unusable, and one "IPO day" is not an IPO day

The `mean` column above peaks at **95x**, which is nonsense. The cause is `PPCB`
(Propanc Biopharma): a sub-penny stock at $0.02 that did a large reverse split, so the
ratio comes out at 12,500x. Two separate problems collapse into that one row:

1. **Reverse splits are not adjusted in this history**, so the "growth" is a corporate
   action, not a return.
2. **`min_date` is not the IPO date for uplistings.** Five names in this universe
   (PPCB, AVBH, ALM, CIIT, CAPS) already traded OTC before moving to NASDAQ, so the
   first row is just the start of the download window in January 2025 — months before the
   offer date.

Re-anchoring entry on the first session **on or after the actual offer date** fixes both.

In [ ]:
offer_dates = ipo_universe[["Symbol", "Offer Date"]].rename(
    columns={"Symbol": "Ticker"}
)
with_offer = stocks_df.merge(offer_dates, on="Ticker")
post_offer = with_offer[with_offer["Date"] >= with_offer["Offer Date"]]
entry_day = post_offer.loc[post_offer.groupby("Ticker")["Date"].idxmin()]

comparison = pd.DataFrame(
    {
        "median (min_date)": ipo_day[GROWTH_COLS].median(),
        "median (offer date)": entry_day[GROWTH_COLS].median(),
        "mean (min_date)": ipo_day[GROWTH_COLS].mean(),
        "mean (offer date)": entry_day[GROWTH_COLS].mean(),
    }
)
print(comparison.round(3).to_string())
print("\nbest month, re-anchored:", entry_day[GROWTH_COLS].median().idxmax())

**Answer 3: 1 month, median growth 0.9354 — the `1` option.**

The honest reading of that number: **the "optimal" holding period is the shortest one
offered, and it still loses 6.5%.** Every horizon is below 1.0, and the median decays
almost monotonically from 0.94 at one month to 0.48 at eleven — hold a median 2025 IPO for
a year and half the capital is gone. The only kink is a small bounce at months 6 and
12, well inside the noise of 130 names.

For an investor the conclusion is not "buy IPOs and sell after a month", it is
**"buying the median IPO at the first close is a losing trade at every horizon, and the
best available outcome from tuning the exit is to lose less"**. The decay shape is what
expect from post-IPO drift: first-day pricing is set by demand in a supported book, and
that support decays as lockups roll off and coverage thins.

Re-anchoring on the true offer date leaves the answer unchanged (month 1, median 0.924)
but repairs the mean column — it drops from 95x to a believable 1.06x, and the medians
shift by 1–5 points. That is the version to trust for anything beyond this homework.

---

## Question 4 — [Strategy] Simple RSI-based trading strategy

> **What is the total profit (in $ thousands) from investing $1000 every time a stock was
> oversold (RSI < 30)?**

In [ ]:
import gdown

FILE_ID = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
PARQUET_PATH = smaz.config.RAW_DIR / "module2_indicators.parquet"

if not PARQUET_PATH.exists():
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", str(PARQUET_PATH), quiet=False)

indicators = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
indicators["Date"] = pd.to_datetime(indicators["Date"])

print(indicators.shape)
print("tickers:", indicators["Ticker"].nunique())
print("date range:", indicators["Date"].min().date(), "->", indicators["Date"].max().date())
indicators[["Date", "Ticker", "Close_x", "rsi", "growth_future_30d", "ticker_type"]].head()

In [ ]:
RSI_THRESHOLD = 30
TRADE_SIZE = 1000

selected_df = indicators[
    (indicators["rsi"] < RSI_THRESHOLD)
    & (indicators["Date"] >= "2000-01-01")
    & (indicators["Date"] <= "2025-06-01")
]

net_income = TRADE_SIZE * (selected_df["growth_future_30d"] - 1).sum()

print(f"trades triggered     : {len(selected_df):,}")
print(f"avg 30-day return    : {(selected_df['growth_future_30d'] - 1).mean():.4%}")
print(f"win rate             : {(selected_df['growth_future_30d'] > 1).mean():.2%}")
print(f"capital deployed     : ${TRADE_SIZE * len(selected_df):,}")
print(f"net income           : ${net_income:,.2f}  ->  ${net_income / 1000:.1f} thousand")

The three sanity checks quoted in the task all reproduce exactly — 5,206 trades, a 1.26%
average 30-day return and a 55.13% win rate — so the dataset and the filter match the
intended ones.

In [ ]:
# where the trades come from, and whether the edge survives outside the biggest bucket
per_market = selected_df.groupby("ticker_type").apply(
    lambda g: pd.Series(
        {
            "trades": len(g),
            "avg_return": (g["growth_future_30d"] - 1).mean(),
            "win_rate": (g["growth_future_30d"] > 1).mean(),
            "net_income": TRADE_SIZE * (g["growth_future_30d"] - 1).sum(),
        }
    ),
    include_groups=False,
)
print(per_market.round(4).to_string())

**Answer 4: $65,806 — the `65` option.**

Reading past the headline:

- **The edge is real but tiny.** 1.26% per 30-day trade at a 55% win rate is a genuine
  mean-reversion signal, but it is an average over $5.2M of deployed capital for $66k of
  profit — a **1.26% return on each ticket**, not 66x anything. Quoting it as a dollar
  total flatters it.
- **Capital is ignored.** The $1,000-per-signal rule assumes unlimited, free capital and
  overlapping positions. 5,206 signals over 25 years means many concurrent open trades;
  the strategy is untradeable as stated without a position-sizing rule.
- **No costs, no slippage.** At 1.26% gross per trade, a round-trip cost of even 20bps on
  an illiquid oversold name eats ~16% of the edge.
- Loosening the threshold from 25 to 30 more than triples the opportunity count
  (~1,568 → 5,206). More signals at a similar per-trade edge is the right direction, but
  it also means RSI < 30 is a *weak* filter — it fires on ~2.3% of all bar-days.

### What horizon is `growth_future_30d` actually measuring?

Before using this column as a benchmark for anything, it is worth checking what it is. The
name says 30 days. The sibling column `growth_30d` is verifiably
`Close / Close.shift(30)` — 30 trading rows — so the natural reading is a 30-day forward
return. That reading is wrong.

In [ ]:
ordered = indicators.sort_values(["Ticker", "Date"])

print("growth_30d == Close_x / Close_x.shift(30)?")
trailing = ordered.groupby("Ticker")["Close_x"].shift(30)
print("   max abs diff:", float((ordered["growth_30d"] - ordered["Close_x"] / trailing).abs().max()))

print("\ngrowth_future_30d vs an h-row FORWARD return:")
for h in (5, 21, 30):
    forward = ordered.groupby("Ticker")["Close_x"].shift(-h) / ordered["Close_x"]
    diff = (ordered["growth_future_30d"] - forward).abs()
    print(f"   h={h:2d}: max abs diff={diff.max():.10f}  rows compared={int(diff.notna().sum()):,}")

**`growth_future_30d` is a 5-trading-day forward return, not a 30-day one** — an exact
match to ten decimal places across all 229,767 rows. The name is simply wrong, and the
column's dispersion confirms it independently: for AAPL its log-standard-deviation is
0.0627 against 0.1582 for a true 30-day forward return, and its median (1.0051) matches the
5-day median exactly.

This does **not** change the answer to Q4 — the arithmetic is unchanged and all three of the
task's sanity checks still reproduce. What it changes is the *interpretation*, and by a
lot: the 1.264% average is earned per **week**, not per month. The strategy holds each
position five sessions, so capital turns over ~50 times a year rather than ~8, and the
annualised rate implied by that mean is **88%**, not ~11%. It makes the Q4 baseline a much
higher bar than its headline suggests — which matters for Q5.

---

## Question 5 — [Exploratory] Beating the Q4 strategy on profit *and* alpha

> Most IPO strategies deliver negative average **and** median returns.
> **How would you change the strategy to increase profitability?**

**Answer: stop buying IPOs, and add a volatility filter to the Q4 signal.**
`RSI < 30 AND natr > 3 AND slowk < 20` earns **+62.7% more profit on the same capital**
with **78.1% annualised alpha against 27.8%**.

Why abandon the IPO book: Q3 already showed the *best* of twelve holding periods still has
a median growth of **0.9354** — every horizon loses for the typical name. Q4's rule has
positive expectancy over 5,206 trades and 25 years, so that is the thing worth improving.

**Two metrics, fixed before searching.**

- **Profit at equal capital.** Q4's own metric (total dollars at $1,000/signal) is
  $1,000 × 5,206 = **$5.206M**.
- **Alpha.** Intercept of `r ~ benchmark`, the benchmark being the equal-weight forward
  5-day return of the same region on the same date, errors clustered by date.

**And one rule of procedure:** the search runs on **2000–2014 only**. The 2015–2025 half is
a holdout that never influences which rule is picked.

In [ ]:
import itertools

import statsmodels.formula.api as smf

PERIODS_PER_YEAR = 252 / 5      # growth_future_30d is a 5-session forward return (shown above)
Q4_CAPITAL = 5_206_000.0        # $1,000 x 5,206 Q4 signals -- the budget every rule is scored on

scan = indicators[
    (indicators["Date"] >= "2000-01-01") & (indicators["Date"] <= "2025-06-01")
].copy()
scan["r"] = scan["growth_future_30d"] - 1
scan = scan.dropna(subset=["r"]).reset_index(drop=True)
# benchmark = equal-weight forward return of the same region on the same date
scan["bench"] = scan.groupby(["Date", "ticker_type"])["r"].transform("mean")

IS = (scan["Date"] < "2015-01-01").to_numpy()    # search here
OOS = ~IS                                        # never used to choose anything


def score(mask, half=None, label="", clustered=False):
    """Profit at equal capital + market-relative alpha for one rule."""
    m = mask if half is None else (mask & half)
    if m.sum() < 30:
        return None
    sel = scan[m]
    r, b = sel["r"], sel["bench"]
    beta = np.cov(r, b, ddof=1)[0, 1] / np.var(b, ddof=1)   # closed-form OLS: cheap at scale
    alpha = r.mean() - beta * b.mean()
    s = r.sort_values()
    out = {
        "strategy": label, "n": len(r), "mean": r.mean(), "win": (r > 0).mean(),
        "sharpe": r.mean() / r.std() * np.sqrt(PERIODS_PER_YEAR),
        "cvar20": s.iloc[: max(1, int(0.2 * len(s)))].mean(), "beta": beta,
        "alpha_ann": (1 + alpha) ** PERIODS_PER_YEAR - 1,
        "profit_equal_cap": Q4_CAPITAL * r.mean(),
    }
    if clustered:   # only worth the cost for finalists
        fit = smf.ols("r ~ bench", data=sel).fit(
            cov_type="cluster", cov_kwds={"groups": sel["Date"]}
        )
        out["alpha_t"] = fit.tvalues["Intercept"]
    return out


rsi30 = (scan["rsi"] < 30).to_numpy()
always = np.ones(len(scan), bool)
print(pd.DataFrame([
    score(rsi30, label="Q4 baseline: rsi<30", clustered=True),
    score(always, label="control: always invested", clustered=True),
]).to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

q_is, q_oos = score(rsi30, IS), score(rsi30, OOS)
print(f"\nQ4 by half --  IS 2000-2014: mean={q_is['mean']:.4f} alpha={q_is['alpha_ann']:.3f}"
      f"  |  OOS 2015-2025: mean={q_oos['mean']:.4f} alpha={q_oos['alpha_ann']:.3f}")

Control: holding everything always gives **beta 1.0000** and a machine-zero intercept, so
the framework has no built-in tilt. **The bar is Q4's $65,806 and 27.8% alpha (t = 6.24).**

In [ ]:
# a library of confirming conditions -- oscillators, trend, volatility, macro, candles, region
scan["natr"] = scan["natr"]              # normalised ATR: average true range as % of price
scan["vol_rel"] = scan["Volume"] / scan.groupby("Ticker")["Volume"].transform(
    lambda s: s.rolling(20, min_periods=5).mean()
)

COND = {}
for col, t in [("slowk", 20), ("slowk", 30), ("slowd", 20), ("fastk", 20), ("fastd", 20),
               ("fastk_rsi", 20), ("willr", -80), ("willr", -90), ("cci", -100), ("cci", -150),
               ("mfi", 20), ("mfi", 15), ("ultosc", 35), ("ultosc", 30), ("cmo", -30), ("cmo", -50),
               ("aroonosc", -50), ("bop", 0), ("trix", 0), ("roc", -5), ("mom", 0), ("apo", 0), ("ppo", 0)]:
    COND[f"{col}<{t}"] = (scan[col] < t).to_numpy()
for col, t in [("adx", 25), ("adx", 30), ("minus_di", 25), ("minus_di", 30), ("dx", 25), ("natr", 3)]:
    COND[f"{col}>{t}"] = (scan[col] > t).to_numpy()
COND["adx<20"] = (scan["adx"] < 20).to_numpy()
COND["yr-up"] = (scan["growth_365d"] > 1).to_numpy()
COND["90d-up"] = (scan["growth_90d"] > 1).to_numpy()
COND[">SMA20"] = (scan["Close_x"] > scan["SMA20"]).to_numpy()
COND[">SMA10"] = (scan["Close_x"] > scan["SMA10"]).to_numpy()
COND["growingMA"] = (scan["growing_moving_average"] == 1).to_numpy()
COND["30d-down"] = (scan["growth_30d"] < 1).to_numpy()
COND["7d-down"] = (scan["growth_7d"] < 1).to_numpy()
COND["1d-up"] = (scan["growth_1d"] > 1).to_numpy()
COND["hivol"] = (scan["volatility"] > scan["volatility"].median()).to_numpy()
COND["lowvol"] = (scan["volatility"] <= scan["volatility"].median()).to_numpy()
COND["vol-spike"] = (scan["vol_rel"] > 1.5).to_numpy()
COND["vol-quiet"] = (scan["vol_rel"] < 1.0).to_numpy()
COND["snp30d-up"] = (scan["growth_snp500_30d"] > 1).to_numpy()
COND["snp365d-up"] = (scan["growth_snp500_365d"] > 1).to_numpy()
COND["snp7d-down"] = (scan["growth_snp500_7d"] < 1).to_numpy()
COND["snp30d-down"] = (scan["growth_snp500_30d"] < 1).to_numpy()
COND["gold30d-up"] = (scan["growth_gold_30d"] > 1).to_numpy()
COND["oil30d-down"] = (scan["growth_wti_oil_30d"] < 1).to_numpy()
COND["fed<3"] = (scan["FEDFUNDS"] < 3).to_numpy()
COND["curve-steep"] = ((scan["DGS10"] - scan["DGS1"]) > 1).to_numpy()
COND["cpi<3"] = (scan["cpi_core_yoy"] < 3).to_numpy()
for col in ["cdlhammer", "cdlinvertedhammer", "cdlengulfing", "cdlmorningstar", "cdlpiercing",
            "cdl3whitesoldiers", "cdldragonflydoji", "cdlbelthold", "cdlharami", "cdlmatchinglow",
            "cdlhomingpigeon", "cdldoji", "cdlspinningtop", "cdllongline"]:
    COND[f"{col}+"] = (scan[col] > 0).to_numpy()
COND["obv-up"] = (scan["obv"] > 0).to_numpy()
COND["adosc>0"] = (scan["adosc"] > 0).to_numpy()
for reg in scan["ticker_type"].unique():
    COND[f"reg={reg}"] = (scan["ticker_type"] == reg).to_numpy()

BASE = {name: (scan[col] < t).to_numpy() for name, (col, t) in {
    "rsi<30": ("rsi", 30), "rsi<25": ("rsi", 25), "rsi<35": ("rsi", 35),
    "willr<-80": ("willr", -80), "mfi<20": ("mfi", 20)}.items()}

# PASS 1 -- score every base x condition on the IN-SAMPLE half only
rows = []
for bname, bmask in BASE.items():
    for cname, cmask in COND.items():
        s = score(bmask & cmask, IS, f"{bname} & {cname}")
        if s and s["n"] >= 300:
            rows.append(s)
p1 = pd.DataFrame(rows)
passed = p1[(p1["mean"] > q_is["mean"]) & (p1["alpha_ann"] > q_is["alpha_ann"])]
print(f"{len(BASE)} bases x {len(COND)} conditions -> {len(p1)} rules with >=300 IS trades; "
      f"{len(passed)} beat Q4 in-sample on both metrics")

# PASS 2 -- take those winners to the holdout. Nothing below influenced the choice above.
oos = pd.DataFrame([score(BASE[s.split(" & ")[0]] & COND[s.split(" & ", 1)[1]], OOS, s)
                    for s in passed["strategy"]])
merged = passed.merge(oos, on="strategy", suffixes=("_is", "_oos"))
survived = merged[(merged["mean_oos"] > q_oos["mean"]) & (merged["alpha_ann_oos"] > q_oos["alpha_ann"])]
show = ["strategy", "n_is", "mean_is", "alpha_ann_is", "n_oos", "mean_oos", "sharpe_oos", "alpha_ann_oos"]
fmt = lambda v: f"{v:,.4f}"
print(f"\n--- {len(survived)} of {len(merged)} survive the holdout (top 6) ---")
print(survived.nlargest(6, "mean_oos")[show].to_string(index=False, float_format=fmt))
print("\n--- best in-sample rules that DIED out of sample ---")
print(merged[~merged["strategy"].isin(survived["strategy"])].nlargest(4, "mean_is")[show]
      .to_string(index=False, float_format=fmt))

**Volatility is the filter the notebook was missing.** Of 264 rules, 46 beat Q4 in-sample
and only **13 survive the holdout** — and the top of that list is not another oscillator,
it is `natr > 3`: take the oversold signal only when average true range exceeds **3% of
the price**. The logic is plain once seen — mean reversion pays in proportion to how far
prices travel, and an oversold reading on a quiet stock has nothing to revert.

The holdout also does its job on the other side. The three best *in-sample* rules —
`rsi<25 & oil30d-down` (3.62%/trade), `rsi<25 & mfi<20` (3.45%), `rsi<25 & reg=INDIA`
(3.40%) — collapse to **+0.19%, −0.93% and −0.90%** after 2015. They were fitted to
2008–09, exactly as `rsi<20` was.

Stacking the survivors is where it lands: adding the old `slowk < 20` confirmation on top
of the volatility filter gives the best combination of edge and trade count.

In [ ]:
# stack the surviving conditions on top of rsi<30 -- still gated on the in-sample half only
surv_conds = sorted({s.split(" & ", 1)[1] for s in survived["strategy"]})
stacked = []
for c1, c2 in itertools.combinations(surv_conds, 2):
    m = BASE["rsi<30"] & COND[c1] & COND[c2]
    s_is, s_oos = score(m, IS, f"rsi<30 & {c1} & {c2}"), score(m, OOS, f"rsi<30 & {c1} & {c2}")
    if s_is and s_oos and s_is["n"] >= 300 and s_oos["n"] >= 100:
        if s_is["mean"] > q_is["mean"] and s_is["alpha_ann"] > q_is["alpha_ann"]:
            stacked.append({**{f"{k}_is": v for k, v in s_is.items()},
                            **{f"{k}_oos": v for k, v in s_oos.items()},
                            "strategy": s_is["strategy"]})
stacked = pd.DataFrame(stacked)
beat = stacked[(stacked["mean_oos"] > q_oos["mean"]) & (stacked["alpha_ann_oos"] > q_oos["alpha_ann"])]
print(f"{len(stacked)} stacked pairs pass the IS gate; {len(beat)} also beat Q4 out of sample\n")
print(beat.nlargest(5, "mean_oos")[show].to_string(index=False, float_format=fmt))

FINAL = {
    "Q4 baseline: rsi<30": rsi30,
    "rsi<30 & slowk<20": BASE["rsi<30"] & COND["slowk<20"],
    "rsi<30 & natr>3": BASE["rsi<30"] & COND["natr>3"],
    "WINNER: rsi<30 & natr>3 & slowk<20": BASE["rsi<30"] & COND["natr>3"] & COND["slowk<20"],
}
final = pd.DataFrame([score(m, label=k, clustered=True) for k, m in FINAL.items()])
print("\n=== full sample 2000-2025 ===")
print(final.to_string(index=False, float_format=fmt))
q = final.set_index("strategy").loc["Q4 baseline: rsi<30"]
print("\nvs Q4 at equal capital:")
for _, row in final.iloc[1:].iterrows():
    print(f"  {row['strategy']:36s} profit {row['profit_equal_cap'] / q['profit_equal_cap'] - 1:+7.1%}"
          f"   alpha {row['alpha_ann'] - q['alpha_ann']:+.3f}   sharpe {row['sharpe'] - q['sharpe']:+.2f}")

In [ ]:
best = BASE["rsi<30"] & COND["natr>3"] & COND["slowk<20"]

print("natr threshold sweep -- a knife-edge would show one good cell, not a gradient:")
for t in [0, 2, 2.5, 3, 3.5, 4, 5]:
    m = BASE["rsi<30"] & COND["slowk<20"] & (scan["natr"] > t).to_numpy()
    si, so = score(m, IS), score(m, OOS)
    print(f"   natr>{t:<4} IS n={si['n']:5d} mean={si['mean']:+.4f} | "
          f"OOS n={so['n']:5d} mean={so['mean']:+.4f} sharpe={so['sharpe']:+.2f} alpha={so['alpha_ann']:+.3f}")

print("\n5-year blocks -- rule vs Q4:")
for lo, hi in [("2000", "2005"), ("2005", "2010"), ("2010", "2015"), ("2015", "2020"), ("2020", "2026")]:
    blk = ((scan["Date"] >= lo) & (scan["Date"] < hi)).to_numpy()
    a, bq = score(best, blk), score(rsi30, blk)
    print(f"   {lo}-{hi}  rule n={a['n']:4d} mean={a['mean']:+.4f} alpha={a['alpha_ann']:+.3f}  |  "
          f"Q4 n={bq['n']:4d} mean={bq['mean']:+.4f} alpha={bq['alpha_ann']:+.3f}  "
          f"{'win' if a['mean'] > bq['mean'] else 'LOSE'}")

print("\nby region, and after round-trip costs:")
for reg in scan["ticker_type"].unique():
    rm = (scan["ticker_type"] == reg).to_numpy()
    a, bq = score(best & rm), score(rsi30 & rm)
    print(f"   {reg:6s} rule n={a['n']:4d} mean={a['mean']:+.4f}  |  Q4 n={bq['n']:4d} mean={bq['mean']:+.4f}")
for bps in [0, 20, 50]:
    net = lambda m: scan.loc[m, "r"].mean() - bps / 10_000
    print(f"   {bps:2d}bp  Q4 ${Q4_CAPITAL * net(rsi30):>9,.0f}   rule ${Q4_CAPITAL * net(best):>9,.0f}")

print("\nfactor attribution -- is it alpha, or a bigger dose of short-term reversal?")
scan["mom_12m"], scan["rev_1m"] = scan["growth_365d"] - 1, scan["growth_30d"] - 1
for label, m in [("rsi<30 & slowk<20", BASE["rsi<30"] & COND["slowk<20"]), ("WINNER", best)]:
    sub = scan[m].dropna(subset=["r", "bench", "mom_12m", "rev_1m"])
    line = []
    for formula in ["r ~ bench", "r ~ bench + mom_12m", "r ~ bench + mom_12m + rev_1m"]:
        fit = smf.ols(formula, data=sub).fit(cov_type="cluster", cov_kwds={"groups": sub["Date"]})
        a = fit.params["Intercept"]
        line.append(f"{(1 + a) ** PERIODS_PER_YEAR - 1:+.3f} (t={fit.tvalues['Intercept']:+.2f})")
    print(f"   {label:20s} mkt {line[0]}   +mom {line[1]}   +reversal {line[2]}")

### Result

| | Q4 `rsi<30` | **`rsi<30 & natr>3 & slowk<20`** |
| --- | --- | --- |
| trades | 5,206 | 1,622 |
| return per trade | 1.264% | **2.058%** |
| **profit at equal capital** | $65,806 | **$107,052 (+62.7%)** |
| **annualised alpha** | 27.8% | **78.1%** |
| alpha t (clustered) | 6.24 | **8.04** |
| annualised Sharpe | 0.95 | **1.48** |
| win rate | 55.1% | **60.4%** |
| CVaR₂₀ | **−0.083** | −0.108 |

It beat the baseline on both target metrics in a half of history it was never fitted to —
**2.36% per trade out of sample against Q4's 0.82%**, Sharpe 1.83, alpha 68.5% — and every
robustness check agrees:

- **Not a knife-edge.** Raising the `natr` cut from 2 → 5 improves the out-of-sample edge
  monotonically (1.24% → 5.56%) while trade count falls. A mined threshold shows one good
  cell, not a gradient; `natr > 3` is chosen as the capacity/edge compromise.
- **Wins in all five 5-year blocks and all three regions.** Most striking is 2020–2026,
  where **Q4 decays to 0.36% per trade and 3.6% alpha while the rule holds 2.11% and 79%** —
  the plain RSI edge is being competed away and the volatility-filtered one is not.
- **Survives costs.** At a 50bp round trip it still earns $81,022 against Q4's $39,776.

**The two honest deductions.**

1. **Worse tail.** CVaR₂₀ goes from −8.3% to −10.8% — a volatility filter buys volatile
   names, so losing trades lose more. The Sharpe and win rate still improve, so this is a
   trade worth making, but it needs smaller position sizes, not the same $1,000 ticket.
2. **It is not alpha in a factor sense.** Against market + momentum the intercept is +86.6%
   (t = 8.72); adding short-term reversal drives it *negative* (−51%, t = −2.88). The rule
   is a **concentrated dose of the reversal premium**, not a new source of return — it buys
   deeper reversal exposure where that premium is largest. (The regression is partly
   circular: `rev_1m` nearly *is* the signal, and a linear factor fit extrapolates badly
   into the tail this rule lives in. The trades really did earn 2.06% each.)

**Recommendation.** Trade **`RSI < 30 AND natr > 3 AND slowk < 20`**, position-sized down
to hold risk per trade constant. Remaining limits: 1,622 trades over 25 years is ~65/year,
so capacity is far below Q4's; the stacking step reused the holdout once; and the whole
test lives on 33 tickers, so re-test outside them before sizing up.

---

## Summary of answers

Copy these into `answers.md` and then into the submission form.

| Q | Question | Computed | Submitted option |
| --- | --- | --- | --- |
| 1 | Total withdrawn value of the largest class | Acquisition Corp, $499.99M | **500** |
| 2 | Median Sharpe ratio on 2026-09-11 | 0.0501 | **0.04** |
| 3 | Holding period maximising median growth | 1 month (median 0.9354) | **1** |
| 4 | Net income from the RSI < 30 strategy | $65,805.59 | **65** |
| 5 | Improving IPO strategy profitability | trade `RSI<30 & natr>3 & slowk<20`: +62.7% profit at equal capital, 78.1% vs 27.8% alpha | *free text* |